# AEM4L5 E03 — cProfile básico

**Objetivo:** dejar de optimizar por intuición y usar evidencia.


## Qué mide cProfile

```mermaid
flowchart LR
    A["run_pipeline"] --> B["normalize_text"]
    A --> C["summarize"]
    A --> D["format_output"]
    B --> E["Hotspot si acumula mucho cumtime"]
```

| Columna | Significado | Cómo usarla |
|---|---|---|
| `ncalls` | Cantidad de llamadas | Detectar repetición excesiva |
| `tottime` | Tiempo propio | Detectar función costosa internamente |
| `cumtime` | Tiempo acumulado | Priorizar hotspots |
| `percall` | Promedio por llamada | Ver costo individual |


## Demo: limpieza lenta vs regex compilada

El objetivo no es memorizar regex, sino ver cómo un profiler cambia la conversación: primero medimos, después optimizamos.


In [ ]:
import cProfile
import io
import pstats
import re

texts = ["Contrato!!! cláusula 12, monto $1000, territorio AR."] * 6000


def clean_text_slow(text: str) -> str:
    result = ""
    for char in text:
        if char.isalnum() or char.isspace():
            result += char.lower()
    return result


compiled_noise = re.compile(r"[^A-Za-zÁÉÍÓÚáéíóúÑñ0-9\s]+")


def clean_text_fast(text: str) -> str:
    return compiled_noise.sub("", text).lower()


def process_batch(func):
    return [func(text) for text in texts]


def profile_call(label, func):
    profiler = cProfile.Profile()
    profiler.enable()
    process_batch(func)
    profiler.disable()
    stream = io.StringIO()
    pstats.Stats(profiler, stream=stream).sort_stats("cumtime").print_stats(6)
    print(f"\n--- {label} ---")
    print(stream.getvalue())

profile_call("lento", clean_text_slow)
profile_call("optimizado", clean_text_fast)


## Cierre docente

Si `cumtime` se concentra en limpieza de texto, la mejora está en CPU. Si el tiempo real se va esperando APIs, cProfile solo cuenta parte de la historia y hay que mirar I/O.
